In [221]:
import numpy as np
import pandas as pd
import yfinance as yf
from edgar import *
from earningscall import get_company
from datetime import datetime,timedelta
import re
import libsql
import os
from dotenv import load_dotenv



In [222]:

load_dotenv()

try:
    # Testing the native driver
    conn = libsql.connect(database=os.getenv("TURSO_URL"), auth_token=os.getenv("TURSO_TOKEN"))
    cursor = conn.cursor()
except Exception as e:
    print(f"❌ Environment Error: {e}")

# Creating sec_mda_risk table

In [45]:
set_identity(os.getenv("SEC_API_USER_AGENT"))

# Financials

In [102]:

def map_msft_fiscal_metadata(filing_date, form_type):
    """
    filing_date_str: 'YYYY-MM-DD'
    form_type: '10-K' or '10-Q'
    
    Returns: (fiscal_year, fiscal_period)
    """
    month = filing_date.month
    year = filing_date.year
    
    if "10-K" in form_type:
        # 10-K is filed after June 30, belongs to that calendar year
        return year, "FY"
    
    elif "10-Q" in form_type:
        # Microsoft Q1 is filed in Oct/Nov (belongs to NEXT fiscal year)
        if month in [10, 11, 12]:
            return year + 1, "Q1"
        # Microsoft Q2 is filed in Jan/Feb
        elif month in [1, 2, 3]:
            return year, "Q2"
        # Microsoft Q3 is filed in Apr/May
        elif month in [4, 5, 6]:
            return year, "Q3"
            
    return None, None

# Example Usage:
# f_year, f_period = map_msft_fiscal_metadata("2025-10-23", "10-Q")
# Result: (2026, "Q1")

def get_core_financials(ticker="MSFT"):
    """
    Extracts financial metrics from XBRL statements.
    """
    company = Company(ticker)
    # We go back to 2018 to get a full 8 years of historical context
    filings = company.get_filings(form=["10-K", "10-Q"]).filter(filing_date="2019-01-01:")
    raw_data = []
    for filing in filings:
        print(f"Processing {filing.form} filed on {filing.filing_date}...")
        f_year, f_period = map_msft_fiscal_metadata(filing.filing_date, filing.form)            
        f_id = f"{ticker}_{f_year}_{f_period}_FILING"
        effective_date = (filing.filing_date + timedelta(days=1)).strftime('%Y-%m-%d')

        report = filing.obj()

        # 1. Income Statement Data
        income_df = report.financials.income_statement().to_dataframe()
        # We look for the most common XBRL tags (Concepts)
        revenue = income_df[income_df['standard_concept']=='Revenue'].iloc[0, 3]
        net_income = income_df[income_df['standard_concept']=='NetIncome'].iloc[0, 3]

        # 2. Cash Flow Data
        cf_df = report.financials.cashflow_statement().to_dataframe()
        op_cash_flow = cf_df[cf_df['label']=='Net cash from operations'].iloc[0, 3]
        capex = cf_df[cf_df['standard_concept']=='CapitalExpenses'].iloc[0, 3]


        #3. Balance sheet
        bal_df = report.financials.balance_sheet().to_dataframe()
        cash_eq = bal_df[bal_df['standard_concept']=='CashAndMarketableSecurities'].iloc[0, 3]
        tot_assets = bal_df[bal_df['standard_concept']=='Assets'].iloc[0, 3]
        tot_liab = bal_df[bal_df['standard_concept']=='Liabilities'].iloc[0, 3]
        tot_eq = bal_df[bal_df['standard_concept']=='AllEquityBalance'].iloc[0, 3]
        
        raw_data.append({
            'filing_id': f_id,
            'ticker': ticker,
            'filing_date': filing.filing_date,
            'effective_date': effective_date, 
            'fiscal_year': f_year,
            'fiscal_period': f_period,
            "revenue": float(revenue),
            "net_income": float(net_income),
            "op_cash_flow": float(op_cash_flow),
            "capex":float(capex),
            "cash_eq":float(cash_eq),
            "total_liability":float(tot_liab),
            "total_assets":float(tot_assets),
            "total_equity":float(tot_eq)}
        )
    return pd.DataFrame(raw_data)     

data = get_core_financials()

Processing 10-Q filed on 2026-01-28...
Processing 10-Q filed on 2025-10-29...
Processing 10-K filed on 2025-07-30...
Processing 10-Q filed on 2025-04-30...
Processing 10-Q filed on 2025-01-29...
Processing 10-Q filed on 2024-10-30...
Processing 10-K filed on 2024-07-30...
Processing 10-Q filed on 2024-04-25...
Processing 10-Q filed on 2024-01-30...
Processing 10-Q filed on 2023-10-24...
Processing 10-K filed on 2023-07-27...
Processing 10-Q filed on 2023-04-25...
Processing 10-Q filed on 2023-01-24...
Processing 10-Q filed on 2022-10-25...
Processing 10-K filed on 2022-07-28...
Processing 10-Q filed on 2022-04-26...
Processing 10-Q filed on 2022-01-25...
Processing 10-Q filed on 2021-10-26...
Processing 10-K filed on 2021-07-29...
Processing 10-Q filed on 2021-04-27...
Processing 10-Q filed on 2021-01-26...
Processing 10-Q filed on 2020-10-27...
Processing 10-K filed on 2020-07-30...
Processing 10-Q filed on 2020-04-29...
Processing 10-Q filed on 2020-01-29...
Processing 10-Q filed on 

In [ ]:
def calculate_Q4(raw_df):
    """
    Takes the raw SEC DataFrame and cleanly isolates Q4 for complete years,
    while safely passing through incomplete years (like 2026).
    """
    final_records = []
    
    # Process sequentially by year
    for year in sorted(raw_df['fiscal_year'].unique()):
        year_data = raw_df[raw_df['fiscal_year'] == year]
        
        # 1. Isolate the pieces
        qs = year_data[year_data['fiscal_period'].isin(['Q1', 'Q2', 'Q3'])]
        fy = year_data[year_data['fiscal_period'] == 'FY']
        
        # 2. Always append the standard quarters we have
        for _, row in qs.iterrows():
            final_records.append(row.to_dict())
            
        # 3. The Strict Guardrail for Q4
        # We only calculate Q4 if we have the FY filing AND we have exactly 3 prior quarters.
        if not fy.empty:
            if len(qs) == 3:
                fy_row = fy.iloc[0].copy()
                
                # Flow Metrics Subtraction
                fy_row['revenue'] = fy_row['revenue'] - qs['revenue'].sum()
                fy_row['net_income'] = fy_row['net_income'] - qs['net_income'].sum()
                fy_row['op_cash_flow'] = fy_row['op_cash_flow'] - qs['op_cash_flow'].sum()
                fy_row['capex'] = fy_row['capex'] - qs['capex'].sum()
                
                # Snapshot Metrics (cash_eq, total_liability, etc.) remain untouched
                
                # Update Identifiers
                fy_row['fiscal_period'] = 'Q4'
                fy_row['filing_id'] = fy_row['filing_id'].replace('FY', 'Q4')
                
                final_records.append(fy_row.to_dict())
            else:
                print(f" Alert: FY {year} is missing a Q1/Q2/Q3 filing. Cannot safely calculate Q4.")
        else:
            # This handles 2026 perfectly - it silently ignores Q4 and moves on
            print(f"ℹ Info: FY {year} is incomplete (No 10-K yet). Skipping Q4 calculation.")

    # Convert back to DataFrame and sort chronologically
    clean_df = pd.DataFrame(final_records)
    clean_df['filing_date'] = clean_df['filing_date'].astype(str)
    clean_df['effective_date'] = clean_df['effective_date'].astype(str)
    clean_df = clean_df.sort_values(['fiscal_year', 'fiscal_period'], ascending=[False, False]).reset_index(drop=True)
    
    return clean_df

# Execute the transformation
# final_financials_df = calculate_Q4(data)

In [115]:
final = calculate_Q4(data)

 Alert: FY 2019 is missing a Q1/Q2/Q3 filing. Cannot safely calculate Q4.
ℹ Info: FY 2026 is incomplete (No 10-K yet). Skipping Q4 calculation.


In [118]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS financial_filings_raw (
    filing_id TEXT PRIMARY KEY,
    ticker TEXT,
    filing_date TEXT,
    effective_date TEXT,
    fiscal_year INTEGER,
    fiscal_period TEXT,
    revenue REAL,
    net_income REAL,
    op_cash_flow REAL,
    capex REAL,
    cash_eq REAL,
    total_liability REAL,
    total_assets REAL,
    total_equity REAL
)
""")
conn.commit()

records_to_insert = final[[
    'filing_id', 'ticker', 'filing_date', 'effective_date', 
    'fiscal_year', 'fiscal_period', 'revenue', 'net_income', 
    'op_cash_flow', 'capex', 'cash_eq', 'total_liability', 
    'total_assets', 'total_equity'
]].values.tolist()

# Execute batch insert
cursor.executemany("""
    INSERT OR REPLACE INTO financial_filings_raw 
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", records_to_insert)

conn.commit()
print(f" Successfully staged {len(records_to_insert)} financial quarters into Turso.")

✅ Successfully staged 28 financial quarters into Turso.


SEC MDA and Risk Sectors

In [ ]:
def semantic_sentence_splitter(text, max_chars=1500):
    """Splits 'wall of text' into chunks at sentence boundaries (~350-400 tokens)."""
    if not text: return []
    # Regex split: period/exclamation/question followed by space
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chars and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += " " + sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

In [ ]:

def isolate_alpha_chapters(raw_mda_text):
    """
    A state-machine router that ruthlessly slices the MD&A, keeping only 
    the high-value volatility drivers and dropping all other noise.
    """
    if not raw_mda_text: return ""
        
    # The 'ON' Switches (High Alpha)
    target_headers = [
        r'Economic Conditions, Challenges, and Risks',
        r'Fiscal Year \d{4} Compared with Fiscal Year \d{4}', # For 10-Ks
        r'Three Months Ended.*?Compared with',                # For 10-Qs
        r'Other Planned Uses of Capital'
    ]
    
    # The 'OFF' Switches (Regulatory Noise & Stale Data)
    stop_headers = [
        r'Metrics',
        r'Six Months Ended',  # Kills Q2 YTD sections
        r'Nine Months Ended', # Kills Q3 YTD sections
        r'OTHER INCOME \(EXPENSE\)',
        r'INCOME TAXES',
        r'NON-GAAP FINANCIAL MEASURES',
        r'CRITICAL ACCOUNTING ESTIMATES',
        r'STATEMENT OF MANAGEMENT'
    ]
    
    target_regex = re.compile("|".join(target_headers), re.IGNORECASE)
    stop_regex = re.compile("|".join(stop_headers), re.IGNORECASE)
    
    lines = raw_mda_text.replace('\r\n', '\n').split('\n')
    extracted_chapters = []
    
    # The State Machine starts OFF. It ignores the opening boilerplate.
    is_reading = False 
    
    for line in lines:
        line_str = line.strip()
        if not line_str: continue
            
        # If we see a target header, flip the switch ON
        if target_regex.search(line_str):
            is_reading = True
            
        # If we see a stop header, flip the switch OFF
        if stop_regex.search(line_str):
            is_reading = False
            
        # If the switch is ON, save the text
        if is_reading:
            extracted_chapters.append(line_str)
            
    return "\n".join(extracted_chapters)



def clean_mda_narrative(raw_text):
    """
    Cleans SEC MD&A text by evaporating fragmented table rows using 
    alphabetic density and word-count thresholds, preserving narrative context.
    """
    if not raw_text: return ""

    # 1. Standardize line breaks and split
    lines = raw_text.replace('\r\n', '\n').split('\n')
    clean_lines = []

    # 2. Blacklist of orphaned table row headers and SEC boilerplate
    debris_blacklist = {
        'three months ended', 'six months ended', 'nine months ended', 'year ended',
        'percentage change', 'in millions', 'in billions', 'unaudited',
        'consolidated statements', 'total revenue', 'gross margin',
        'operating income', 'net income', 'operating expenses', 'cost of revenue',
        'as a percent of revenue', 'percentagechange', 'diluted earnings per share'
    }

    # 3. Regex to catch dynamic headers
    comparison_header_regex = r'(?:Fiscal Year|Three Months Ended|Six Months Ended).*?Compared with'

    # 4. The Main Processing Loop
    for line in lines:
        line_str = line.strip()
        
        # Preserve natural paragraph spacing
        if not line_str:
            clean_lines.append("")
            continue

        # Filter A: Is it known table debris?
        if line_str.lower() in debris_blacklist: 
            continue
            
        # Filter B: Is it a repeating comparative header?
        if re.search(comparison_header_regex, line_str, re.IGNORECASE):
            continue

        # Filter C: The "Table Cell" Density Test
        total_chars = len(line_str)
        alpha_chars = sum(c.isalpha() for c in line_str)
        if total_chars > 0 and (alpha_chars / total_chars) < 0.5:
            continue 

        # Filter D: The "Ghost Table Row" Check
        if len(line_str.split()) < 5:
            continue

        # If it survives all filters, it is high-value narrative.
        clean_lines.append(line_str)

    # 5. Reassembly
    text = "\n".join(clean_lines)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()

    return text

In [ ]:
def process_filing_to_chunks(filing, ticker, f_date):
    """
    The master extraction function. Routes SEC text, cleans it, chunks it, 
    and prepares the exact tuples needed for the Turso database.
    """
    try:
        doc = filing.obj()
    except Exception as e:
        print(f"Failed to download/parse filing object: {e}")
        return []

    form = filing.form
    doc_id = f"{ticker}_{form.replace('-', '')}_{str(f_date).replace('-', '')}"
    
    # 1. Safely handle the edgartools API differences
    mda_raw, risk_raw = "", ""
    if form == "10-K":
        mda_raw = doc.management_discussion
        risk_raw = doc.risk_factors
    elif form == "10-Q":
        mda_raw = doc['Part I, Item 2']
        
        risk_raw = doc['Part II, Item 1A']

    else: 
        print(f"Unsupported form type: {form}. Skipping.")
    chunks_to_db = []

    # ==========================================
    # PIPELINE 1: The MD&A
    # ==========================================
    if mda_raw:
        # Route out the accounting and YTD garbage
        routed_mda = isolate_alpha_chapters(mda_raw)
        
        # Evaporate the tables
        clean_mda_narrative(routed_mda)
        
        # Chunk it for FinBERT
        mda_chunks = semantic_sentence_splitter(clean_mda_narrative, max_chars=1500)
        
        for i, content in enumerate(mda_chunks):
            chunk_id = f"{doc_id}_MDA_{i}"
            # Format matches Turso: (chunk_id, doc_id, ticker, filing_date, item_type, index, content, score)
            chunks_to_db.append((chunk_id, doc_id, ticker, str(f_date), 'MDA', i, content, 0.0))

    # ==========================================
    # PIPELINE 2: Risk Factors
    # ==========================================
    if risk_raw:
        # The "10-Q Alpha Check"
        # If it's a 10-Q and it's short, it usually just says "no material changes."
        if form == "10-Q" and len(risk_raw) < 300 and "no material changes" in risk_raw.lower():
            # We save one single chunk to tell the model things are stable
            chunks_to_db.append((f"{doc_id}_RISK_0", doc_id, ticker, str(f_date), 'RiskFactorsUpdate', 0, "No material changes to risk factors.", 0.0))
        else:
            # Basic cleanup (Risk factors rarely have tables, so we just fix spacing)
            clean_risk = re.sub(r'\s+', ' ', risk_raw).strip()
            
            # Chunk it for FinBERT
            risk_chunks = semantic_sentence_splitter(clean_risk, max_chars=1500)
            
            item_label = 'RiskFactors' if form == '10-K' else 'RiskFactorsUpdate'
            
            for i, content in enumerate(risk_chunks):
                chunk_id = f"{doc_id}_{item_label}_{i}"
                chunks_to_db.append((chunk_id, doc_id, ticker, str(f_date), item_label, i, content, 0.0))

    return chunks_to_db

In [201]:
filings = Company("MSFT").get_filings(form=["10-K",'10-Q']).filter(filing_date="2018-01-01:")
all_db_rec = []
for filing in filings:
    chunk = process_filing_to_chunks( filing, "MSFT", filing.filing_date)
    all_db_rec.extend(chunk)

In [203]:
len(all_db_rec)

1924

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS sec_chunks (
    chunk_id TEXT PRIMARY KEY,
    doc_id TEXT,
    ticker TEXT,
    filing_date TEXT,
    item_type TEXT,
    chunk_index INTEGER,
    content TEXT,
    sentiment_score REAL
)
""")
conn.commit()
print("✅ Text chunk table successfully created.")

In [202]:
cursor.executemany("""
            INSERT OR REPLACE INTO sec_chunks 
            (chunk_id, doc_id, ticker, filing_date, item_type, chunk_index, content, sentiment_score) 
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, all_db_rec)
conn.commit()


    # Earnings call transcripts

In [40]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS transcripts (
    transcript_id TEXT PRIMARY KEY,
    ticker TEXT,
    event_timestamp TEXT,
    effective_date TEXT,
    fiscal_year INTEGER,
    fiscal_period TEXT,
    content_prepped TEXT,
    content_qa TEXT,
    sentiment_score REAL DEFAULT 0.0
)
""")

In [42]:
import earningscall
from datetime import datetime, timedelta
import json
# --- HARD CONSTANTS ---
START_DATE = datetime(2019, 1, 1)
END_DATE = datetime(2026, 4, 22) # Current Date
def process_msft_history(cursor, ticker="MSFT"):
    company = earningscall.get_company(ticker)
    
    # 1. Define the 5-year boundary
    
    # Storage for all processed quarters
    historical_data = []

    # 2. Iterate through events
    for event in company.events():
        # Skip future events
        event_date = event.conference_date.replace(tzinfo=None)
        if not (START_DATE <= event_date <= END_DATE):
            continue

        # 2. EFFECTIVE MARKET DATE (T+1 Logic)
        # If call is 4:00 PM (16:00) or later, the market reacts the NEXT day
        # if event.conference_date.hour >= 16:
        effective_date = (event.conference_date + timedelta(days=1)).strftime('%Y-%m-%d')
        # else:
            # effective_date = event.conference_date.strftime('%Y-%m-%d')
        print(f"Ingesting: Q{event.quarter} {event.year} (Date: {event.conference_date.strftime('%Y-%m-%d')})")
        
        # 3. FISCAL ALIGNMENT
        # We use the library's year/quarter as the "Source of Truth" for labeling
        f_year = event.year
        f_period = f"Q{event.quarter}" if event.quarter < 4 else "FY"
        t_id = f"{ticker}_{f_year}_{f_period}_TRANSCRIPT"
        
        try:
            # Fetch Level 2 to get speaker names and text
            transcript = company.get_transcript(event=event, level=2)
            
            if not transcript or not transcript.speakers:
                continue

            # 3. Apply the split logic to create our lists of dicts
            prepped, qa = split_transcript_by_turns(transcript.speakers)
            
            # Store results with metadata for the training table
            historical_data.append((t_id,
                ticker,
                event.conference_date.strftime('%Y-%m-%d %H:%M:%S'), # Real Timestamp
                effective_date,                                      # Trading Day 0
                f_year,
                f_period,
                json.dumps(prepped),
                json.dumps(qa)))

        except Exception as e:
            print(f"Error processing Q{event.quarter} {event.year}: {e}")
        
        if historical_data:
            
            cursor.executemany("""
                INSERT OR REPLACE INTO transcripts 
                (transcript_id, ticker, event_timestamp, effective_date, fiscal_year, fiscal_period, content_prepped, content_qa) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, historical_data)
        print(f"\n✅ Total Ingested: {len(historical_data)} filings.")
        conn.commit()
    return historical_data

def split_transcript_by_turns(turns):
    """
    Processes Level 2 turns into two lists of dictionaries based on the Q&A pivot.
    """
    prepped_list = []
    qa_list = []
    is_qa_started = False
    
    # MSFT specific pivot markers
    qa_markers = ["q and a", "q&a","q and a"]

    for turn in turns:
        speaker_name = turn.speaker_info.name if turn.speaker else "Unknown"
        speaker_title = turn.speaker_info.title if turn.speaker else "Unknown" 
        text = turn.text if turn.text else ""
        text_lower = text.lower()

        # Detect transition
        if not is_qa_started:
            if any(marker in text_lower for marker in qa_markers):
                is_qa_started = True
                # We skip the specific turn that announces the Q&A to keep data clean
                continue 

        # Add to respective list as a dictionary
        entry = {"speaker_name": speaker_name,'speaker_title':speaker_title, "text": text}
        
        if is_qa_started:
            qa_list.append(entry)
        else:
            prepped_list.append(entry)

    return prepped_list, qa_list

# Execute
msft_dataset = process_msft_history(cursor)

Ingesting: Q2 2026 (Date: 2026-01-28)

✅ Total Ingested: 1 filings.
Ingesting: Q1 2026 (Date: 2025-10-29)

✅ Total Ingested: 2 filings.
Ingesting: Q4 2025 (Date: 2025-07-30)

✅ Total Ingested: 3 filings.
Ingesting: Q3 2025 (Date: 2025-04-30)

✅ Total Ingested: 4 filings.
Ingesting: Q2 2025 (Date: 2025-01-29)

✅ Total Ingested: 5 filings.
Ingesting: Q1 2025 (Date: 2024-10-30)

✅ Total Ingested: 6 filings.
Ingesting: Q4 2024 (Date: 2024-07-30)

✅ Total Ingested: 7 filings.
Ingesting: Q3 2024 (Date: 2024-04-25)

✅ Total Ingested: 8 filings.
Ingesting: Q2 2024 (Date: 2024-01-30)

✅ Total Ingested: 9 filings.
Ingesting: Q1 2024 (Date: 2023-10-24)

✅ Total Ingested: 10 filings.
Ingesting: Q4 2023 (Date: 2023-07-25)

✅ Total Ingested: 11 filings.
Ingesting: Q3 2023 (Date: 2023-04-25)

✅ Total Ingested: 12 filings.
Ingesting: Q2 2023 (Date: 2023-01-24)

✅ Total Ingested: 13 filings.
Ingesting: Q1 2023 (Date: 2022-10-25)

✅ Total Ingested: 14 filings.
Ingesting: Q4 2022 (Date: 2022-07-26)

✅ To

In [218]:
    data = yf.download(['MSFT','^VIX'], start="2018-01-01", progress=False)


Market Data 

In [220]:
data['Low']['MSFT'].head()

Date
2018-01-02    78.457430
2018-01-03    78.888730
2018-01-04    79.439308
2018-01-05    80.228449
2018-01-08    80.384441
Name: MSFT, dtype: float64

In [204]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS market_data (
    trading_date TEXT PRIMARY KEY,
    msft_close REAL,
    msft_volume INTEGER,
    msft_return REAL,
    vix_close REAL,
    tnx_yield REAL
)
""")
conn.commit()
print("✅ Market Data table created.")

✅ Market Data table created.


In [224]:
# One-time schema update
try:
    cursor.execute("ALTER TABLE market_data ADD COLUMN qqq_close REAL")
    cursor.execute("ALTER TABLE market_data ADD COLUMN spy_close REAL")
    conn.commit()
    print("✅ Database updated with OHLC columns.")
except:
    print("Columns likely already exist.")

✅ Database updated with OHLC columns.


In [229]:
import yfinance as yf
import pandas as pd
import numpy as np

def ingest_market_data(cursor,conn):
    print("Fetching Market and Macro Data from Yahoo Finance...")
    
    # Define our tickers and the 7-year timeframe
    tickers = [ "MSFT", "^VIX", "^TNX", "QQQ", "SPY" ]
    start_date = "2018-01-01"
    end_date = "2026-04-30" # Adjust to current date
    
    # Download data
    data = yf.download(tickers, start=start_date, end=end_date, progress=False)
    
    # Clean up multi-index columns if using newer yfinance versions
    
    closes = data['Close'].copy()
    volumes = data['Volume']['MSFT'].copy() # We only need MSFT volume
    open = data['Open']['MSFT'].copy()
    high = data['High']['MSFT'].copy()
    low = data['Low']['MSFT'].copy()
    
    # Create a clean DataFrame
    df = pd.DataFrame(index=closes.index)
    df['msft_close'] = closes['MSFT']
    df['msft_volume'] = volumes
    df['msft_open'] = open
    df['msft_high'] = high
    df['msft_low'] = low
    df['vix_close'] = closes['^VIX']
    df['tnx_yield'] = closes['^TNX']
    df['qqq_close'] = closes['QQQ']
    df['spy_close'] = closes['SPY']
    
    # Calculate MSFT Daily Log Return (Critical feature for quant models)
    df['msft_return'] = np.log(df['msft_close'] / df['msft_close'].shift(1))
    
    # Drop rows with NaNs (like the first row due to the shift, or market holidays)
    df = df.dropna()
    
    # Prepare for Turso ingestion
    records = []
    for date, row in df.iterrows():
        # Format date to standard string YYYY-MM-DD
        date_str = date.strftime('%Y-%m-%d')
        
        records.append((
            date_str, 
            float(row['msft_close']), 
            int(row['msft_volume']), 
            float(row['msft_open']),
            float(row['msft_high']),
            float(row['msft_low']),
            float(row['msft_return']), 
            float(row['vix_close']), 
            float(row['tnx_yield']),
            float(row['qqq_close']),
            float(row['spy_close'])
        ))
        
    # Batch Insert
    print(f"Pushing {len(records)} trading days to Turso...")
    cursor.executemany("""
        INSERT OR REPLACE INTO market_data 
        (trading_date, msft_close, msft_volume, msft_open, msft_high, msft_low, msft_return, vix_close, tnx_yield, qqq_close, spy_close) 
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)
    
    conn.commit()
    print("✅ Market and Macro data successfully staged!")

# Execute
# ingest_market_data(cursor)

In [230]:
ingest_market_data(cursor,conn)

Fetching Market and Macro Data from Yahoo Finance...
Pushing 2088 trading days to Turso...
✅ Market and Macro data successfully staged!
